In [1]:
from pathlib import Path

import pandas as pd

DATA_PATH = Path("data/updated_result_with_BERT_eval_120.csv")
TARGET_COLUMN = "Diagnosis Category"
FEATURE_COLUMNS = [
    "age",
    "gender_numeric",
    "symptoms",
    "Patient History",
    "Remarks",
    "treatment",
    "timespan",
    "Diagnosis",
]

df = pd.read_csv(DATA_PATH)

X = df[FEATURE_COLUMNS].copy()
Y = df[TARGET_COLUMN].copy()

print(f"X shape: {X.shape}")
print(f"Y shape: {Y.shape}")
print(f"Y classes: {Y.nunique()}")
X.head()

X shape: (120, 8)
Y shape: (120,)
Y classes: 5


,age,gender_numeric,symptoms,Patient History,Remarks,treatment,timespan,Diagnosis
0,53,0,"Lower back pain, Stiff neck, Numb feeling in e...",Regular physiotherapy sessions.,Further evaluation needed.,Electrical stimulation treatments,"3-6 months: Strengthening, functional training...",spinal tumors
1,45,1,"Aching joints, Puffiness in joint areas, Muscl...",Similar condition diagnosed 4 years ago.,"all fine, no fresh complain",Pain management with medications,"3-6 months: Strengthening, functional training...",osteoarthritis
2,56,1,"Generalized discomfort, Mobility issues, Pain ...",No significant medical history.,No additional notes.,Customized rehabilitation programs,"3-6 months: Strengthening, functional training...",cobb angle estimation
3,26,1,"Discomfort in groin, Limited mobility, Challen...","The patient reports bilateral hip pain, worsen...",Routine check-up required.,Lifestyle adjustments to reduce symptoms,"3-6 months: Strengthening, functional training...",Avascular Necrosis of Bilateral Hips
4,70,1,"Discomfort in groin, Hip stiffness, Trouble wi...",MRI indicates avascular necrosis in both hips....,Routine check-up required.,Lifestyle adjustments to reduce symptoms,"3-6 months: Strengthening, functional training...",Avascular Necrosis of Bilateral Hips


In [2]:
import json
import sys
from pathlib import Path

import httpx
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_openai import ChatOpenAI


def _ensure_backend_on_path() -> Path:
    candidates = [
        Path.cwd(),
        Path.cwd().parent,
        Path.cwd() / "backend",
        Path.cwd().parent / "backend",
        Path("/home/yuyaoli/personal/ISE547/backend"),
    ]
    for candidate in candidates:
        if (candidate / "app").exists():
            candidate_str = str(candidate)
            if candidate_str not in sys.path:
                sys.path.insert(0, candidate_str)
            return candidate
    raise ModuleNotFoundError("Cannot locate backend root containing app package.")


BACKEND_ROOT = _ensure_backend_on_path()
from app.core.config import settings

# Infermedica endpoint config (from app config)
INFERMEDICA_BASE_URL = settings.INFERMEDICA_BASE_URL
INFERMEDICA_APP_ID = settings.INFERMEDICA_APP_ID
INFERMEDICA_APP_KEY = settings.INFERMEDICA_APP_KEY
INFERMEDICA_LANGUAGE = settings.INFERMEDICA_LANGUAGE
PARSE_PATH = "/parse"
DIAGNOSIS_PATH = "/diagnosis"

# LLM config (from app config)
OPENROUTER_BASE_URL = settings.OPENROUTER_BASE_URL
OPENROUTER_API_KEY = settings.OPENROUTER_API_KEY
DEFAULT_CHAT_MODEL = settings.DEFAULT_CHAT_MODEL

LLM_SYSTEM_PROMPT = """
You are a clinical triage assistant.

You will receive:
1) Basic patient information (age)
2) Patient symptoms and medical history text (symptoms + Patient History)
3) Output from a structured diagnostic interface (/diagnosis output JSON)

Please output:
- A concise, structured diagnostic conclusion;
- 1-3 most likely disease directions, highlighting the supporting rationale for each;
- Clear risk warnings (e.g., emergency "red flag" signals);
- Recommended next steps;
- A mandatory disclaimer stating that this is not a final clinical diagnosis.

Constraints:
- Base your response strictly on the provided input; do not fabricate facts not present in the text;
- Output in English.
""".strip()


def gender_numeric_to_sex(gender_numeric: int) -> str:
    if int(gender_numeric) == 0:
        return "male"
    if int(gender_numeric) == 1:
        return "female"
    return "other"


def build_parse_text(age: int, symptoms: str, patient_history: str) -> str:
    return (
        f"Patient age: {age}. "
        f"Symptoms: {symptoms}. "
        f"Patient history: {patient_history}."
    )


def infermedica_headers() -> dict[str, str]:
    return {
        "App-Id": INFERMEDICA_APP_ID,
        "App-Key": INFERMEDICA_APP_KEY,
        "Accept-Language": INFERMEDICA_LANGUAGE,
        "Content-Type": "application/json",
    }


def call_parse(client: httpx.Client, *, text: str, age: int, sex: str) -> dict:
    payload = {
        "text": text,
        "age": {"value": int(age)},
        "sex": sex,
    }
    response = client.post(
        f"{INFERMEDICA_BASE_URL.rstrip('/')}{PARSE_PATH}",
        headers=infermedica_headers(),
        json=payload,
    )
    response.raise_for_status()
    return response.json()


def parse_mentions_to_evidence(parse_result: dict) -> list[dict[str, str]]:
    mentions = parse_result.get("mentions", []) or []
    evidence: list[dict[str, str]] = []
    for mention in mentions:
        mention_id = str(mention.get("id", "")).strip()
        choice_id = str(mention.get("choice_id", "")).strip()
        if mention_id and choice_id:
            evidence.append({"id": mention_id, "choice_id": choice_id})
    return evidence


def call_diagnosis(
    client: httpx.Client,
    *,
    age: int,
    sex: str,
    evidence: list[dict[str, str]],
) -> dict:
    payload = {
        "sex": sex,
        "age": {"value": int(age)},
        "evidence": evidence,
    }
    response = client.post(
        f"{INFERMEDICA_BASE_URL.rstrip('/')}{DIAGNOSIS_PATH}",
        headers=infermedica_headers(),
        json=payload,
    )
    response.raise_for_status()
    return response.json()


def build_llm_user_prompt(
    *,
    age: int,
    symptoms: str,
    patient_history: str,
    diagnosis_payload: dict,
) -> str:
    return f"""
Patient information:
- Age: {age}
- Symptoms: {symptoms}
- Patient History: {patient_history}

/diagnosis output (JSON):
{json.dumps(diagnosis_payload, ensure_ascii=False)}

Task:
Please provide a final diagnosis summary in English based on the information above.
""".strip()


def create_llm() -> ChatOpenAI:
    return ChatOpenAI(
        model=DEFAULT_CHAT_MODEL,
        api_key=OPENROUTER_API_KEY,
        base_url=OPENROUTER_BASE_URL,
        temperature=0.2,
    )


def run_llm_diagnosis(
    llm: ChatOpenAI,
    *,
    age: int,
    symptoms: str,
    patient_history: str,
    diagnosis_payload: dict,
) -> str:
    user_prompt = build_llm_user_prompt(
        age=age,
        symptoms=symptoms,
        patient_history=patient_history,
        diagnosis_payload=diagnosis_payload,
    )
    messages = [
        SystemMessage(content=LLM_SYSTEM_PROMPT),
        HumanMessage(content=user_prompt),
    ]
    response = llm.invoke(messages)
    return str(response.content).strip()

In [3]:
# 控制只跑前 N 条数据
RUN_TOP_N = 1

sample_df = df.head(RUN_TOP_N).copy()
llm = create_llm()
rows: list[dict] = []

with httpx.Client(timeout=30.0) as client:
    for row_idx, row in sample_df.iterrows():
        age = int(row["age"])
        symptoms = str(row["symptoms"])
        patient_history = str(row["Patient History"])
        sex = gender_numeric_to_sex(int(row["gender_numeric"]))

        input_text = build_parse_text(
            age=age,
            symptoms=symptoms,
            patient_history=patient_history,
        )

        parse_result = call_parse(
            client,
            text=input_text,
            age=age,
            sex=sex,
        )
        evidence = parse_mentions_to_evidence(parse_result)

        diagnosis_result = call_diagnosis(
            client,
            age=age,
            sex=sex,
            evidence=evidence,
        )

        llm_final_output = run_llm_diagnosis(
            llm,
            age=age,
            symptoms=symptoms,
            patient_history=patient_history,
            diagnosis_payload=diagnosis_result,
        )

        rows.append(
            {
                "row_index": row_idx,
                "age": age,
                "symptoms": symptoms,
                "patient_history": patient_history,
                "input_text": input_text,
                "target_y": row[TARGET_COLUMN],
                "ground_truth_diagnosis": row["Diagnosis"],
                "parse_mentions_count": len(parse_result.get("mentions", []) or []),
                "parse_output_json": json.dumps(parse_result, ensure_ascii=False),
                "diagnosis_output_json": json.dumps(diagnosis_result, ensure_ascii=False),
                "llm_final_output": llm_final_output,
            }
        )

        print(
            f"row={row_idx} done, mentions={len(evidence)}, "
            f"llm_len={len(llm_final_output)}"
        )

results_df = pd.DataFrame(rows)
results_df.head()

row=0 done, mentions=5, llm_len=3310


,row_index,age,symptoms,patient_history,input_text,target_y,parse_mentions_count,parse_output_json,diagnosie_output_json,llm_final_output
0,0,53,"Lower back pain, Stiff neck, Numb feeling in e...",Regular physiotherapy sessions.,"Patient age: 53. Symptoms: Lower back pain, St...",Spinal disorders,5,"{""mentions"": [{""id"": ""s_1190"", ""name"": ""Back p...","{""question"": {""type"": ""group_single"", ""text"": ...",**Diagnostic Conclusion**\nThe clinical presen...


In [4]:
OUTPUT_CSV_PATH = Path(f"data/eval_agent_top{RUN_TOP_N}_results.csv")
results_df.to_csv(OUTPUT_CSV_PATH, index=False, encoding="utf-8-sig")

print(f"Saved: {OUTPUT_CSV_PATH.resolve()}")
print(f"Rows: {len(results_df)}")
OUTPUT_CSV_PATH

Saved: /home/yuyaoli/personal/ISE547/backend/scripts/data/eval_agent_top1_results.csv
Rows: 1


PosixPath('data/eval_agent_top1_results.csv')